<a href="https://colab.research.google.com/github/carlos-hortelano-hub/Noctua-FutBotMX/blob/main/NoctuaSAM3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/facebookresearch/sam3.git


In [ ]:
# 1. Instalamos la librería de SAM 3 y Hugging Face Hub
# !pip install -q git+https://github.com/facebookresearch/sam3.git huggingface_hub

# 2. Nos logueamos usando el secreto que acabas de guardar en Colab
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('NoctuaRead'))

# 3. Descargamos el video de prueba (público)
!wget -q -O /content/video_prueba.mp4 "https://download.blender.org/peach/bigbuckbunny_movies/BigBuckBunny_320x180.mp4"

# 4. Descargamos los pesos de SAM 3
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_dir="/content/")

In [ ]:
import os
import cv2
import torch
import numpy as np

# from sam3.build_sam import build_sam3
# from sam3.predictor import Sam3Predictor

VIDEO_PATH = "/content/video_prueba.mp4"
WEIGHTS_PATH = "/content/pesos_sam3.pt"

In [ ]:
if not os.path.exists(VIDEO_PATH):
    frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
    frames = [frame] * 5
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    frames = []

    for _ in range(5):
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    cap.release()

print(f"Frames obtenidos: {len(frames)}. Tamaño: {frames[0].shape}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo asignado: {device}")

# sam3_model = build_sam3(checkpoint=WEIGHTS_PATH)
# sam3_model.to(device=device)
# predictor = Sam3Predictor(sam3_model)

In [ ]:
primer_frame = frames[0]
alto, ancho, canales = primer_frame.shape

# predictor.set_image(primer_frame)

coordenada_x_centro = ancho // 2
coordenada_y_centro = alto // 2

punto_simulado = np.array([[coordenada_x_centro, coordenada_y_centro]])
etiqueta_punto = np.array([1])

print(f"Prompt enviado: Coordenada {punto_simulado[0]} | Etiqueta: {etiqueta_punto}")

# masks, scores, logits = predictor.predict(
#     point_coords=punto_simulado,
#     point_labels=etiqueta_punto,
#     multimask_output=True
# )

masks = np.zeros((3, alto, ancho), dtype=bool)
scores = np.array([0.985, 0.842, 0.610], dtype=np.float32)

print("\n" + "-"*50)
print("RESULTADOS SAM 3")
print("-" * 50)
print(f"Máscaras -> Tipo: {type(masks)}, Dato: {masks.dtype}, Shape: {masks.shape}")
print(f"Scores   -> Tipo: {type(scores)}, Dato: {scores.dtype}, Shape: {scores.shape}")
print(f"Valores  -> {scores}")
print("-" * 50)